# DeepFilterNet-Light — Lightweight Speech Enhancement

A lightweight DeepFilterNet-inspired model for MVDR post-enhancement.

**Key Features:**
- **< 1M parameters** for real-time smartphone deployment
- **Dual-path architecture**: ERB band processing + Deep Filtering
- **GRU-based temporal modeling** for capturing long-range dependencies
- **SDR loss only** for cleaner optimization
- **Learning rate scheduling** (warmup + cosine annealing) to avoid early plateau
- **Target: SNR > 15 dB**

In [2]:
import os
import re
import glob
import math
import json
import shutil
from dataclasses import dataclass
from typing import Optional, Tuple, List

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Dataset

from tqdm.auto import tqdm

try:
    import torchaudio
    _HAS_TORCHAUDIO = True
except ModuleNotFoundError:
    torchaudio = None
    _HAS_TORCHAUDIO = False

try:
    import soundfile as sf
    _HAS_SOUNDFILE = True
except ModuleNotFoundError:
    sf = None
    _HAS_SOUNDFILE = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print('Device:', DEVICE)
print('Torch:', torch.__version__)

Device: cpu
Torch: 2.10.0


## ERB (Equivalent Rectangular Bandwidth) Utilities

DeepFilterNet processes audio in ERB frequency bands for perceptually-motivated enhancement.

In [3]:
def hz_to_erb(hz: float) -> float:
    """Convert Hz to ERB scale."""
    return 9.265 * math.log(1 + hz / (24.7 * 9.265))


def erb_to_hz(erb: float) -> float:
    """Convert ERB scale to Hz."""
    return 24.7 * 9.265 * (math.exp(erb / 9.265) - 1)


def create_erb_filterbank(
    n_fft: int,
    sr: int,
    n_erb_bands: int,
    min_freq: float = 20.0,
    max_freq: Optional[float] = None,
) -> Tuple[torch.Tensor, torch.Tensor, List[Tuple[int, int]]]:
    """Create ERB filterbank matrices for analysis and synthesis.
    
    Returns:
        erb_fb: [n_erb_bands, n_freqs] analysis filterbank
        erb_fb_inv: [n_freqs, n_erb_bands] synthesis filterbank  
        band_indices: list of (start, end) frequency bin indices per band
    """
    if max_freq is None:
        max_freq = sr / 2
    
    n_freqs = n_fft // 2 + 1
    freqs = torch.linspace(0, sr / 2, n_freqs)
    
    # ERB band edges
    min_erb = hz_to_erb(min_freq)
    max_erb = hz_to_erb(max_freq)
    erb_edges = torch.linspace(min_erb, max_erb, n_erb_bands + 1)
    hz_edges = torch.tensor([erb_to_hz(e) for e in erb_edges.tolist()])
    
    # Create triangular filterbank
    erb_fb = torch.zeros(n_erb_bands, n_freqs)
    band_indices = []
    
    for i in range(n_erb_bands):
        low = hz_edges[i]
        center = (hz_edges[i] + hz_edges[i + 1]) / 2
        high = hz_edges[i + 1]
        
        # Find frequency bin range
        low_idx = max(0, int((low / (sr / 2)) * (n_freqs - 1)))
        high_idx = min(n_freqs - 1, int((high / (sr / 2)) * (n_freqs - 1)) + 1)
        band_indices.append((low_idx, high_idx))
        
        for j in range(low_idx, high_idx + 1):
            if j < n_freqs:
                freq = freqs[j]
                if freq <= center and center > low:
                    erb_fb[i, j] = (freq - low) / (center - low + 1e-8)
                elif freq > center and high > center:
                    erb_fb[i, j] = (high - freq) / (high - center + 1e-8)
    
    # Normalize rows
    erb_fb = erb_fb / (erb_fb.sum(dim=1, keepdim=True) + 1e-8)
    
    # Pseudo-inverse for synthesis
    erb_fb_inv = erb_fb.T / (erb_fb.sum(dim=0, keepdim=True).T + 1e-8)
    
    return erb_fb, erb_fb_inv, band_indices

## STFT Utilities

In [4]:
# STFT parameters optimized for 16kHz speech
N_FFT = 512
HOP_LENGTH = 128
WIN_LENGTH = 512


def get_window(win_length: int, device: torch.device) -> torch.Tensor:
    """Get sqrt-Hann window for STFT/iSTFT."""
    return torch.sqrt(torch.hann_window(win_length, periodic=True, device=device))


def stft(wav: torch.Tensor, device: torch.device = None) -> torch.Tensor:
    """Compute STFT. Input: [B, T] or [T]. Output: complex [B, F, T] or [F, T]."""
    if device is None:
        device = wav.device
    window = get_window(WIN_LENGTH, device)
    
    squeeze = wav.dim() == 1
    if squeeze:
        wav = wav.unsqueeze(0)
    
    spec = torch.stft(
        wav,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
        window=window,
        center=True,
        return_complex=True,
    )
    
    if squeeze:
        spec = spec.squeeze(0)
    return spec


def istft(spec: torch.Tensor, device: torch.device = None, length: int = None) -> torch.Tensor:
    """Compute iSTFT. Input: complex [B, F, T] or [F, T]. Output: [B, T] or [T]."""
    if device is None:
        device = spec.device
    window = get_window(WIN_LENGTH, device)
    
    squeeze = spec.dim() == 2
    if squeeze:
        spec = spec.unsqueeze(0)
    
    wav = torch.istft(
        spec,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
        window=window,
        center=True,
        length=length,
    )
    
    if squeeze:
        wav = wav.squeeze(0)
    return wav

## DeepFilterNet-Light Model

Lightweight architecture inspired by DeepFilterNet:
- **ERB Encoder**: Processes magnitude in ERB bands
- **Temporal GRU**: Captures long-range dependencies efficiently
- **Deep Filter Module**: Generates multi-frame complex filters
- **< 1M parameters** target

In [5]:
class GroupedLinear(nn.Module):
    """Grouped linear layer for parameter efficiency."""
    def __init__(self, in_features: int, out_features: int, groups: int = 1, bias: bool = True):
        super().__init__()
        assert in_features % groups == 0 and out_features % groups == 0
        self.groups = groups
        self.in_per_group = in_features // groups
        self.out_per_group = out_features // groups
        
        self.weight = nn.Parameter(torch.randn(groups, self.out_per_group, self.in_per_group) * 0.02)
        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [..., in_features]
        batch_shape = x.shape[:-1]
        x = x.view(*batch_shape, self.groups, self.in_per_group)
        x = torch.einsum('...gi,goi->...go', x, self.weight)
        x = x.view(*batch_shape, -1)
        if self.bias is not None:
            x = x + self.bias
        return x


class ConvGLU(nn.Module):
    """Conv1D with Gated Linear Unit activation."""
    def __init__(self, in_ch: int, out_ch: int, kernel_size: int = 3, groups: int = 1):
        super().__init__()
        self.conv = nn.Conv1d(in_ch, out_ch * 2, kernel_size, padding=kernel_size // 2, groups=groups)
        self.out_ch = out_ch
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv(x)
        x, gate = x.chunk(2, dim=1)
        return x * torch.sigmoid(gate)


class ERBEncoder(nn.Module):
    """Encode magnitude spectrum to ERB band features."""
    def __init__(
        self,
        n_freqs: int,
        n_erb_bands: int,
        hidden_dim: int,
        n_layers: int = 2,
    ):
        super().__init__()
        self.n_erb_bands = n_erb_bands
        
        # Learnable ERB projection (more flexible than fixed filterbank)
        self.erb_proj = nn.Linear(n_freqs, n_erb_bands)
        
        # Feature extraction
        layers = []
        in_dim = n_erb_bands
        for i in range(n_layers):
            out_dim = hidden_dim if i == n_layers - 1 else n_erb_bands * 2
            layers.append(ConvGLU(in_dim, out_dim, kernel_size=3))
            layers.append(nn.BatchNorm1d(out_dim))
            in_dim = out_dim
        self.layers = nn.Sequential(*layers)
    
    def forward(self, mag: torch.Tensor) -> torch.Tensor:
        # mag: [B, Freq, T]
        # Apply log compression
        mag = torch.log1p(mag * 10)  # Learnable compression
        
        # Project to ERB bands: [B, Freq, T] -> [B, T, Freq] -> [B, T, E] -> [B, E, T]
        erb = self.erb_proj(mag.transpose(1, 2)).transpose(1, 2)
        
        # Extract features
        return self.layers(erb)


class TemporalGRU(nn.Module):
    """Efficient GRU for temporal modeling."""
    def __init__(self, input_dim: int, hidden_dim: int, num_layers: int = 2, dropout: float = 0.1):
        super().__init__()
        self.gru = nn.GRU(
            input_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=False,  # Causal for real-time
        )
        self.proj = nn.Linear(hidden_dim, input_dim) if hidden_dim != input_dim else nn.Identity()
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, C, T] -> [B, T, C]
        x = x.transpose(1, 2)
        x, _ = self.gru(x)
        x = self.proj(x)
        return x.transpose(1, 2)


class DeepFilterModule(nn.Module):
    """Generate complex deep filters for multi-frame filtering."""
    def __init__(
        self,
        hidden_dim: int,
        n_freqs: int,
        df_order: int = 3,  # Number of past frames to use
        df_bins: int = 96,  # Number of DF frequency bins (lower freqs)
    ):
        super().__init__()
        self.df_order = df_order
        self.df_bins = df_bins
        self.n_freqs = n_freqs
        
        # Deep filter coefficients: 2 (real/imag) * df_order * df_bins
        df_coef_dim = 2 * df_order * df_bins
        
        # Gain for all frequency bins
        self.gain_proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, n_freqs),
            nn.Sigmoid(),
        )
        
        # Deep filter coefficients for low frequencies
        self.df_proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, df_coef_dim),
        )
        
        # Initialize for identity-like behavior
        nn.init.zeros_(self.df_proj[-1].weight)
        nn.init.zeros_(self.df_proj[-1].bias)
    
    def forward(
        self,
        features: torch.Tensor,
        spec: torch.Tensor,
    ) -> torch.Tensor:
        """Apply deep filtering.
        
        Args:
            features: [B, C, T] hidden features
            spec: [B, Freq, T] complex input spectrum
        
        Returns:
            enhanced: [B, Freq, T] complex enhanced spectrum
        """
        B, n_freq, T = spec.shape
        
        # features: [B, C, T] -> [B, T, C]
        feat = features.transpose(1, 2)
        
        # Compute gains: [B, T, Freq]
        gains = self.gain_proj(feat)  # [B, T, Freq]
        gains = gains.transpose(1, 2)  # [B, Freq, T]
        
        # Compute DF coefficients: [B, T, 2 * df_order * df_bins]
        df_coefs = self.df_proj(feat)
        df_coefs = df_coefs.view(B, T, 2, self.df_order, self.df_bins)
        df_real = df_coefs[:, :, 0]  # [B, T, df_order, df_bins]
        df_imag = df_coefs[:, :, 1]
        
        # Apply gain to full spectrum
        enhanced = spec * gains
        
        # Apply deep filtering to low frequencies
        # Pad spectrum for causal filtering
        spec_padded = F.pad(spec[:, :self.df_bins, :], (self.df_order - 1, 0))
        
        # Multi-frame filtering
        df_out = torch.zeros(B, self.df_bins, T, dtype=spec.dtype, device=spec.device)
        
        for k in range(self.df_order):
            # Get frame at offset k
            frame = spec_padded[:, :, self.df_order - 1 - k:self.df_order - 1 - k + T]  # [B, df_bins, T]
            
            # Complex multiplication with DF coefficients
            coef_r = df_real[:, :, k, :].transpose(1, 2)  # [B, df_bins, T]
            coef_i = df_imag[:, :, k, :].transpose(1, 2)
            
            frame_r = frame.real
            frame_i = frame.imag
            
            # (a + bi)(c + di) = (ac - bd) + (ad + bc)i
            out_r = coef_r * frame_r - coef_i * frame_i
            out_i = coef_r * frame_i + coef_i * frame_r
            
            df_out = df_out + torch.complex(out_r, out_i)
        
        # Blend DF output with gain-only output for low frequencies
        # Use softer blending to allow DF to contribute
        alpha = 0.5  # Blend factor
        enhanced[:, :self.df_bins, :] = alpha * df_out + (1 - alpha) * enhanced[:, :self.df_bins, :]
        
        return enhanced


class DeepFilterNetLight(nn.Module):
    """Lightweight DeepFilterNet for real-time speech enhancement.
    
    Target: < 1M parameters
    """
    def __init__(
        self,
        n_fft: int = 512,
        n_erb_bands: int = 32,
        hidden_dim: int = 64,
        gru_dim: int = 96,
        gru_layers: int = 2,
        df_order: int = 3,
        df_bins: int = 96,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.n_fft = n_fft
        self.n_freqs = n_fft // 2 + 1
        
        # ERB encoder
        self.erb_encoder = ERBEncoder(
            n_freqs=self.n_freqs,
            n_erb_bands=n_erb_bands,
            hidden_dim=hidden_dim,
            n_layers=2,
        )
        
        # Temporal modeling
        self.temporal_gru = TemporalGRU(
            input_dim=hidden_dim,
            hidden_dim=gru_dim,
            num_layers=gru_layers,
            dropout=dropout,
        )
        
        # Deep filter module
        self.deep_filter = DeepFilterModule(
            hidden_dim=hidden_dim,
            n_freqs=self.n_freqs,
            df_order=df_order,
            df_bins=min(df_bins, self.n_freqs),
        )
        
        # Parameter count check
        n_params = sum(p.numel() for p in self.parameters())
        print(f"DeepFilterNetLight: {n_params / 1e6:.3f}M parameters")
    
    def forward(self, spec: torch.Tensor) -> torch.Tensor:
        """Forward pass.
        
        Args:
            spec: [B, Freq, T] complex input spectrum
        
        Returns:
            enhanced: [B, Freq, T] complex enhanced spectrum
        """
        # Get magnitude
        mag = torch.abs(spec)
        
        # ERB encoding
        features = self.erb_encoder(mag)  # [B, hidden_dim, T]
        
        # Temporal modeling
        features = self.temporal_gru(features)  # [B, hidden_dim, T]
        
        # Deep filtering
        enhanced = self.deep_filter(features, spec)
        
        return enhanced

## Dataset

In [6]:
def _load_wav(path: str, target_sr: int) -> torch.Tensor:
    """Load audio file and resample if needed."""
    if _HAS_SOUNDFILE:
        data, sr = sf.read(path, dtype="float32", always_2d=True)
        if sr != target_sr:
            if _HAS_TORCHAUDIO:
                wav = torch.from_numpy(np.asarray(data).T)
                wav = torchaudio.functional.resample(wav, sr, target_sr)
            else:
                raise ValueError(f"Sample rate mismatch: {sr} vs {target_sr}")
        else:
            wav = torch.from_numpy(np.asarray(data).T)
    elif _HAS_TORCHAUDIO:
        wav, sr = torchaudio.load(path)
        if sr != target_sr:
            wav = torchaudio.functional.resample(wav, sr, target_sr)
        wav = wav.to(torch.float32)
    else:
        raise ModuleNotFoundError("Install soundfile or torchaudio")
    
    # Mix to mono
    if wav.dim() == 2 and wav.shape[0] > 1:
        wav = wav.mean(dim=0)
    elif wav.dim() == 2:
        wav = wav.squeeze(0)
    
    return wav


class MVDRDataset(Dataset):
    """Dataset for noisy-to-clean pairs (same filenames in both directories)."""
    def __init__(
        self,
        noisy_dir: str,
        clean_dir: str,
        sample_rate: int = 16000,
    ):
        self.noisy_dir = noisy_dir
        self.clean_dir = clean_dir
        self.sr = sample_rate
        
        # Get list of noisy files
        self.files = sorted([f for f in os.listdir(noisy_dir) if f.lower().endswith(".wav")])
        
        # Verify all pairs exist
        missing = []
        for f in self.files:
            clean_path = os.path.join(clean_dir, f)
            if not os.path.exists(clean_path):
                missing.append(f)
        
        if missing:
            raise FileNotFoundError(f"Missing clean files for: {missing[:5]}{'...' if len(missing) > 5 else ''}")
        
        print(f"Loaded {len(self.files)} audio pairs from {noisy_dir}")
    
    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        filename = self.files[idx]
        noisy_path = os.path.join(self.noisy_dir, filename)
        clean_path = os.path.join(self.clean_dir, filename)
        
        noisy_wav = _load_wav(noisy_path, self.sr)
        clean_wav = _load_wav(clean_path, self.sr)
        
        # Match lengths
        min_len = min(noisy_wav.shape[-1], clean_wav.shape[-1])
        noisy_wav = noisy_wav[:min_len]
        clean_wav = clean_wav[:min_len]
        
        # Normalize
        scale = noisy_wav.std() + 1e-8
        noisy_wav = noisy_wav / scale
        clean_wav = clean_wav / scale
        
        return noisy_wav, clean_wav

## SDR Loss Function

Using only SDR (Signal-to-Distortion Ratio) as requested for cleaner optimization.

In [7]:
def sdr_loss(est: torch.Tensor, ref: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """Scale-Invariant SDR loss (negative SDR for minimization).
    
    Args:
        est: Estimated signal [B, T]
        ref: Reference signal [B, T]
    
    Returns:
        loss: Negative SI-SDR (lower is better)
    """
    # Zero-mean
    ref = ref - ref.mean(dim=-1, keepdim=True)
    est = est - est.mean(dim=-1, keepdim=True)
    
    # Compute SI-SDR
    dot = torch.sum(est * ref, dim=-1, keepdim=True)
    s_target = (dot / (torch.sum(ref ** 2, dim=-1, keepdim=True) + eps)) * ref
    e_noise = est - s_target
    
    si_sdr = torch.sum(s_target ** 2, dim=-1) / (torch.sum(e_noise ** 2, dim=-1) + eps)
    si_sdr_db = 10.0 * torch.log10(si_sdr + eps)
    
    return -si_sdr_db.mean()

## Training Configuration

In [8]:
# ==================== CONFIGURATION ====================

# Dataset paths (pre-split directories)
TRAIN_NOISY_DIR = '/Users/emonchowdhury/Desktop/Phase 2/av_zoom/DATASET/prepared_dataset/train/noisy/'
TRAIN_CLEAN_DIR = '/Users/emonchowdhury/Desktop/Phase 2/av_zoom/DATASET/prepared_dataset/train/clean/'
VAL_NOISY_DIR = '/Users/emonchowdhury/Desktop/Phase 2/av_zoom/DATASET/prepared_dataset/val/noisy/'
VAL_CLEAN_DIR = '/Users/emonchowdhury/Desktop/Phase 2/av_zoom/DATASET/prepared_dataset/val/clean/'
TEST_NOISY_DIR = '/Users/emonchowdhury/Desktop/Phase 2/av_zoom/DATASET/prepared_dataset/test/noisy/'
TEST_CLEAN_DIR = '/Users/emonchowdhury/Desktop/Phase 2/av_zoom/DATASET/prepared_dataset/test/clean/'

SAMPLE_RATE = 16000

# Training hyperparameters
BATCH_SIZE = 16
EPOCHS = 150  # More epochs to avoid early plateau
BASE_LR = 3e-4
MIN_LR = 1e-6
WEIGHT_DECAY = 1e-4

# Learning rate schedule
WARMUP_EPOCHS = 5
LR_SCHEDULE = 'cosine'  # 'cosine' or 'plateau'

SEED = 42
NUM_WORKERS = 4 if DEVICE == 'cuda' else 0

# Segment length for training
SEGMENT_SECONDS = 2.0
SEGMENT_SAMPLES = int(SAMPLE_RATE * SEGMENT_SECONDS)

# Checkpointing
CHECKPOINT_DIR = 'checkpoints_dfnet'
CHECKPOINT_PREFIX = 'dfnet'
SAVE_EVERY_EPOCHS = 1
AUTO_RESUME = True

# Early stopping
EARLY_STOPPING = True
EARLY_STOP_PATIENCE = 20  # More patience
EARLY_STOP_MIN_DELTA_DB = 0.05

# ========================================================

## Training Utilities

In [9]:
def _seed_everything(seed: int):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _train_collate(batch):
    """Collate with random cropping for training."""
    mvdr_list, clean_list = zip(*batch)
    mvdr_out, clean_out = [], []
    
    for mvdr, clean in zip(mvdr_list, clean_list):
        length = mvdr.shape[-1]
        if length >= SEGMENT_SAMPLES:
            start = torch.randint(0, length - SEGMENT_SAMPLES + 1, (1,)).item()
            mvdr = mvdr[start:start + SEGMENT_SAMPLES]
            clean = clean[start:start + SEGMENT_SAMPLES]
        else:
            pad = SEGMENT_SAMPLES - length
            mvdr = F.pad(mvdr, (0, pad))
            clean = F.pad(clean, (0, pad))
        
        mvdr_out.append(mvdr)
        clean_out.append(clean)
    
    return torch.stack(mvdr_out), torch.stack(clean_out)


def _val_collate(batch):
    """Collate with center cropping for validation."""
    mvdr_list, clean_list = zip(*batch)
    mvdr_out, clean_out = [], []
    
    for mvdr, clean in zip(mvdr_list, clean_list):
        length = mvdr.shape[-1]
        if length >= SEGMENT_SAMPLES:
            start = (length - SEGMENT_SAMPLES) // 2
            mvdr = mvdr[start:start + SEGMENT_SAMPLES]
            clean = clean[start:start + SEGMENT_SAMPLES]
        else:
            pad = SEGMENT_SAMPLES - length
            mvdr = F.pad(mvdr, (0, pad))
            clean = F.pad(clean, (0, pad))
        
        mvdr_out.append(mvdr)
        clean_out.append(clean)
    
    return torch.stack(mvdr_out), torch.stack(clean_out)


def make_loaders():
    """Create data loaders from pre-split directories."""
    # Create datasets for each split
    train_dataset = MVDRDataset(TRAIN_NOISY_DIR, TRAIN_CLEAN_DIR, SAMPLE_RATE)
    val_dataset = MVDRDataset(VAL_NOISY_DIR, VAL_CLEAN_DIR, SAMPLE_RATE)
    test_dataset = MVDRDataset(TEST_NOISY_DIR, TEST_CLEAN_DIR, SAMPLE_RATE)
    
    print(f'Dataset sizes: train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}')
    
    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=(DEVICE == 'cuda'),
        collate_fn=_train_collate,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=(DEVICE == 'cuda'),
        collate_fn=_val_collate,
    )
    test_loader = DataLoader(
        test_dataset, batch_size=1, shuffle=False,
        num_workers=NUM_WORKERS,
    )
    
    return train_loader, val_loader, test_loader

## Learning Rate Scheduler

Warmup + Cosine Annealing to avoid early plateau.

In [10]:
class WarmupCosineScheduler:
    """Learning rate scheduler with warmup and cosine annealing."""
    def __init__(
        self,
        optimizer: optim.Optimizer,
        warmup_epochs: int,
        total_epochs: int,
        base_lr: float,
        min_lr: float,
    ):
        self.optimizer = optimizer
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs
        self.base_lr = base_lr
        self.min_lr = min_lr
        self.current_epoch = 0
    
    def step(self, epoch: int = None):
        if epoch is not None:
            self.current_epoch = epoch
        else:
            self.current_epoch += 1
        
        if self.current_epoch <= self.warmup_epochs:
            # Linear warmup
            lr = self.base_lr * (self.current_epoch / self.warmup_epochs)
        else:
            # Cosine annealing
            progress = (self.current_epoch - self.warmup_epochs) / (self.total_epochs - self.warmup_epochs)
            lr = self.min_lr + 0.5 * (self.base_lr - self.min_lr) * (1 + math.cos(math.pi * progress))
        
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr
        
        return lr
    
    def get_lr(self):
        return self.optimizer.param_groups[0]['lr']

## Training Loop

In [11]:
def _make_pbar(iterable, desc: str):
    return tqdm(iterable, desc=desc, leave=True, miniters=1, dynamic_ncols=True)


def forward_pass(model, mvdr_wav, clean_wav):
    """Forward pass through model."""
    # STFT
    mvdr_spec = stft(mvdr_wav, DEVICE)  # [B, F, T]
    
    # Model forward
    enhanced_spec = model(mvdr_spec)
    
    # iSTFT
    enhanced_wav = istft(enhanced_spec, DEVICE, length=mvdr_wav.shape[-1])
    
    # Compute SDR loss
    loss = sdr_loss(enhanced_wav, clean_wav)
    
    # Compute baseline SDR for comparison
    with torch.no_grad():
        baseline_loss = sdr_loss(mvdr_wav, clean_wav)
    
    return loss, baseline_loss, enhanced_wav


def train_one_epoch(model, optimizer, train_loader, epoch_idx):
    model.train()
    
    total_loss = 0.0
    total_baseline = 0.0
    n = 0
    
    pbar = _make_pbar(train_loader, desc=f'Train {epoch_idx:03d}')
    for mvdr_wav, clean_wav in pbar:
        mvdr_wav = mvdr_wav.to(DEVICE)
        clean_wav = clean_wav.to(DEVICE)
        
        loss, baseline_loss, _ = forward_pass(model, mvdr_wav, clean_wav)
        
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        
        total_loss += loss.item()
        total_baseline += baseline_loss.item()
        n += 1
        
        enh_sdr = -total_loss / n
        base_sdr = -total_baseline / n
        imp = enh_sdr - base_sdr
        
        pbar.set_postfix(
            sdr=f'{enh_sdr:.2f}dB',
            base=f'{base_sdr:.2f}dB',
            imp=f'{imp:.2f}dB',
            lr=f'{optimizer.param_groups[0]["lr"]:.1e}',
        )
    
    return total_loss / n, total_baseline / n


@torch.no_grad()
def validate(model, val_loader, epoch_idx):
    model.eval()
    
    total_loss = 0.0
    total_baseline = 0.0
    n = 0
    
    pbar = _make_pbar(val_loader, desc=f'Val   {epoch_idx:03d}')
    for mvdr_wav, clean_wav in pbar:
        mvdr_wav = mvdr_wav.to(DEVICE)
        clean_wav = clean_wav.to(DEVICE)
        
        loss, baseline_loss, _ = forward_pass(model, mvdr_wav, clean_wav)
        
        total_loss += loss.item()
        total_baseline += baseline_loss.item()
        n += 1
        
        enh_sdr = -total_loss / n
        base_sdr = -total_baseline / n
        imp = enh_sdr - base_sdr
        
        pbar.set_postfix(
            sdr=f'{enh_sdr:.2f}dB',
            base=f'{base_sdr:.2f}dB',
            imp=f'{imp:.2f}dB',
        )
    
    return total_loss / n, total_baseline / n

## Checkpointing

In [12]:
def save_checkpoint(epoch: int, model: nn.Module, optimizer: optim.Optimizer, scheduler, best_sdr: float):
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    
    payload = {
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_epoch': scheduler.current_epoch,
        'best_sdr': best_sdr,
    }
    
    epoch_path = os.path.join(CHECKPOINT_DIR, f'{CHECKPOINT_PREFIX}_epoch_{epoch:03d}.pt')
    latest_path = os.path.join(CHECKPOINT_DIR, f'{CHECKPOINT_PREFIX}_latest.pt')
    
    torch.save(payload, epoch_path)
    torch.save(payload, latest_path)
    
    return epoch_path


def find_latest_checkpoint():
    latest_path = os.path.join(CHECKPOINT_DIR, f'{CHECKPOINT_PREFIX}_latest.pt')
    if os.path.isfile(latest_path):
        return latest_path
    
    pattern = os.path.join(CHECKPOINT_DIR, f'{CHECKPOINT_PREFIX}_epoch_*.pt')
    candidates = glob.glob(pattern)
    if not candidates:
        return None
    
    best = None
    best_epoch = -1
    for p in candidates:
        m = re.search(r'_epoch_(\d+)\.pt$', os.path.basename(p))
        if m:
            ep = int(m.group(1))
            if ep > best_epoch:
                best_epoch = ep
                best = p
    return best


def maybe_resume(model, optimizer, scheduler):
    if not AUTO_RESUME:
        return 1, -float('inf')
    
    ckpt_path = find_latest_checkpoint()
    if ckpt_path is None:
        return 1, -float('inf')
    
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.current_epoch = ckpt.get('scheduler_epoch', 0)
    best_sdr = ckpt.get('best_sdr', -float('inf'))
    last_epoch = ckpt['epoch']
    
    print(f'Resumed from {ckpt_path} (epoch {last_epoch}, best_sdr={best_sdr:.2f}dB)')
    return last_epoch + 1, best_sdr

## Main Training Function

In [13]:
def main():
    _seed_everything(SEED)
    
    # Create data loaders
    train_loader, val_loader, test_loader = make_loaders()
    
    # Create model
    model = DeepFilterNetLight(
        n_fft=N_FFT,
        n_erb_bands=32,
        hidden_dim=64,
        gru_dim=96,
        gru_layers=2,
        df_order=3,
        df_bins=96,
        dropout=0.1,
    ).to(DEVICE)
    
    # Optimizer with weight decay
    optimizer = optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
    
    # Learning rate scheduler
    scheduler = WarmupCosineScheduler(
        optimizer,
        warmup_epochs=WARMUP_EPOCHS,
        total_epochs=EPOCHS,
        base_lr=BASE_LR,
        min_lr=MIN_LR,
    )
    
    # Resume if checkpoint exists
    start_epoch, best_val_sdr = maybe_resume(model, optimizer, scheduler)
    
    print(f'\nDevice: {DEVICE}')
    print(f'Train/Val/Test: {len(train_loader.dataset)}/{len(val_loader.dataset)}/{len(test_loader.dataset)}')
    print(f'Start epoch: {start_epoch} / Total: {EPOCHS}')
    print(f'Best val SDR so far: {best_val_sdr:.2f}dB')
    
    if start_epoch > EPOCHS:
        print('Training already complete.')
        return
    
    epochs_no_improve = 0
    
    for epoch in range(start_epoch, EPOCHS + 1):
        # Update learning rate
        lr = scheduler.step(epoch)
        
        # Train
        tr_loss, tr_base = train_one_epoch(model, optimizer, train_loader, epoch)
        
        # Validate
        va_loss, va_base = validate(model, val_loader, epoch)
        
        # Compute metrics
        tr_sdr = -tr_loss
        va_sdr = -va_loss
        tr_base_sdr = -tr_base
        va_base_sdr = -va_base
        tr_imp = tr_sdr - tr_base_sdr
        va_imp = va_sdr - va_base_sdr
        
        print(
            f'Epoch {epoch:03d} | lr={lr:.1e} | '
            f'train: sdr={tr_sdr:.2f}dB, imp={tr_imp:.2f}dB | '
            f'val: sdr={va_sdr:.2f}dB, imp={va_imp:.2f}dB'
        )
        
        # Save checkpoint
        if epoch % SAVE_EVERY_EPOCHS == 0:
            path = save_checkpoint(epoch, model, optimizer, scheduler, best_val_sdr)
            print(f'Saved: {path}')
        
        # Early stopping
        if EARLY_STOPPING:
            if va_sdr > best_val_sdr + EARLY_STOP_MIN_DELTA_DB:
                best_val_sdr = va_sdr
                epochs_no_improve = 0
                # Save best model
                best_path = os.path.join(CHECKPOINT_DIR, f'{CHECKPOINT_PREFIX}_best.pt')
                torch.save(model.state_dict(), best_path)
                print(f'New best model: {va_sdr:.2f}dB')
            else:
                epochs_no_improve += 1
            
            if epochs_no_improve >= EARLY_STOP_PATIENCE:
                print(f'Early stopping at epoch {epoch}. Best val SDR: {best_val_sdr:.2f}dB')
                break
    
    print(f'\nTraining complete. Best val SDR: {best_val_sdr:.2f}dB')
    return test_loader

## Inference

In [14]:
def _save_wav(path: str, wav: torch.Tensor, sr: int):
    """Save wav tensor to file."""
    os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
    wav = wav.detach().cpu().to(torch.float32).view(-1)
    
    if _HAS_SOUNDFILE:
        sf.write(path, wav.numpy(), sr)
    elif _HAS_TORCHAUDIO:
        torchaudio.save(path, wav.unsqueeze(0), sr)
    else:
        raise ModuleNotFoundError("Install soundfile or torchaudio")


def load_model_from_checkpoint(checkpoint_path: str, device: str = DEVICE) -> nn.Module:
    """Load model from checkpoint."""
    model = DeepFilterNetLight(
        n_fft=N_FFT,
        n_erb_bands=32,
        hidden_dim=64,
        gru_dim=96,
        gru_layers=2,
        df_order=3,
        df_bins=96,
    ).to(device)
    
    ckpt = torch.load(checkpoint_path, map_location=device)
    state = ckpt['model_state'] if 'model_state' in ckpt else ckpt
    model.load_state_dict(state)
    model.eval()
    return model


@torch.no_grad()
def enhance_waveform(wav: torch.Tensor, model: nn.Module, device: str = DEVICE) -> torch.Tensor:
    """Enhance a mono waveform."""
    if wav.dim() != 1:
        wav = wav.view(-1)
    
    wav = wav.to(device)
    scale = wav.std() + 1e-8
    wav_norm = wav / scale
    
    # STFT
    spec = stft(wav_norm.unsqueeze(0), device)
    
    # Enhance
    enhanced_spec = model(spec)
    
    # iSTFT
    enhanced = istft(enhanced_spec, device, length=wav_norm.shape[-1]).squeeze(0)
    
    return enhanced * scale


def enhance_file(wav_path: str, checkpoint_path: str, output_dir: str, suffix: str = '_enhanced') -> str:
    """Enhance a single wav file."""
    model = load_model_from_checkpoint(checkpoint_path)
    wav = _load_wav(wav_path, SAMPLE_RATE)
    enhanced = enhance_waveform(wav, model)
    
    base = os.path.splitext(os.path.basename(wav_path))[0]
    out_path = os.path.join(output_dir, f'{base}{suffix}.wav')
    _save_wav(out_path, enhanced, SAMPLE_RATE)
    return out_path


def enhance_directory(input_dir: str, checkpoint_path: str, output_dir: str, pattern: str = '*.wav'):
    """Enhance all wav files in a directory."""
    model = load_model_from_checkpoint(checkpoint_path)
    paths = sorted(glob.glob(os.path.join(input_dir, pattern)))
    
    if not paths:
        raise FileNotFoundError(f'No files matching {pattern} in {input_dir}')
    
    os.makedirs(output_dir, exist_ok=True)
    
    for p in tqdm(paths, desc='Enhancing'):
        wav = _load_wav(p, SAMPLE_RATE)
        enhanced = enhance_waveform(wav, model)
        
        base = os.path.splitext(os.path.basename(p))[0]
        out_path = os.path.join(output_dir, f'{base}_enhanced.wav')
        _save_wav(out_path, enhanced, SAMPLE_RATE)
    
    print(f'Saved enhanced files to: {output_dir}')

In [19]:
# Example usage:
checkpoint_path = '/Users/emonchowdhury/Desktop/Phase 2/av_zoom/audio/models/custom_model_2/checkpoints_dfnet/dfnet_epoch_122.pt'
enhance_directory('/Users/emonchowdhury/Desktop/Phase 2/av_zoom/DATASET/prepared_dataset/test/noisy', checkpoint_path, './deepfilternet_enhanced_output')

DeepFilterNetLight: 0.207M parameters


Enhancing:   1%|▏         | 143/9543 [00:08<09:07, 17.18it/s]


KeyboardInterrupt: 

## Test Set Evaluation

In [ ]:
@torch.no_grad()
def evaluate_test_set(checkpoint_path: str, noisy_dir: str = TEST_NOISY_DIR, clean_dir: str = TEST_CLEAN_DIR):
    """Evaluate model on test set."""
    # Create test dataset
    test_dataset = MVDRDataset(noisy_dir, clean_dir, SAMPLE_RATE)
    print(f'Evaluating on {len(test_dataset)} test samples')
    
    model = load_model_from_checkpoint(checkpoint_path)
    
    results = []
    total_base_sdr = 0.0
    total_enh_sdr = 0.0
    
    for idx in tqdm(range(len(test_dataset)), desc='Testing'):
        noisy_wav, clean_wav = test_dataset[idx]
        noisy_wav = noisy_wav.to(DEVICE)
        clean_wav = clean_wav.to(DEVICE)
        
        # Baseline SDR
        base_loss = sdr_loss(noisy_wav.unsqueeze(0), clean_wav.unsqueeze(0))
        base_sdr = -base_loss.item()
        
        # Enhanced SDR
        enhanced = enhance_waveform(noisy_wav, model)
        enh_loss = sdr_loss(enhanced.unsqueeze(0), clean_wav.unsqueeze(0))
        enh_sdr = -enh_loss.item()
        
        results.append({
            'file': test_dataset.files[idx],
            'base_sdr': base_sdr,
            'enh_sdr': enh_sdr,
            'improvement': enh_sdr - base_sdr,
        })
        
        total_base_sdr += base_sdr
        total_enh_sdr += enh_sdr
    
    n = len(results)
    avg_base = total_base_sdr / n
    avg_enh = total_enh_sdr / n
    avg_imp = avg_enh - avg_base
    
    print(f'\n{"="*50}')
    print(f'TEST SET RESULTS ({n} samples)')
    print(f'{"="*50}')
    print(f'  Baseline SDR:  {avg_base:.2f} dB')
    print(f'  Enhanced SDR:  {avg_enh:.2f} dB')
    print(f'  Improvement:   {avg_imp:.2f} dB')
    print(f'{"="*50}')
    
    return {
        'n_samples': n,
        'avg_baseline_sdr': avg_base,
        'avg_enhanced_sdr': avg_enh,
        'avg_improvement': avg_imp,
        'per_sample': results,
    }

In [ ]:
# Example: Evaluate test set
# results = evaluate_test_set('checkpoints_dfnet/dfnet_best.pt')

# Run training

In [ ]:
test_loader = main()

Loaded 76340 audio pairs from /Users/emonchowdhury/Desktop/Phase 2/av_zoom/DATASET/prepared_dataset/train/noisy/
Loaded 9542 audio pairs from /Users/emonchowdhury/Desktop/Phase 2/av_zoom/DATASET/prepared_dataset/val/noisy/
Loaded 9543 audio pairs from /Users/emonchowdhury/Desktop/Phase 2/av_zoom/DATASET/prepared_dataset/test/noisy/
Dataset sizes: train=76340, val=9542, test=9543
DeepFilterNetLight: 0.207M parameters

Device: cpu
Train/Val/Test: 76340/9542/9543
Start epoch: 1 / Total: 150
Best val SDR so far: -infdB


Train 001:   2%|▏         | 88/4772 [00:19<17:29,  4.46it/s, base=0.37dB, imp=-1.49dB, lr=6.0e-05, sdr=-1.11dB]


KeyboardInterrupt: 